# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the metadata and explore dataset records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, their `@id`s, fields, and columns.

In [ ]:
# Identify available record sets and their fields.
pp = pprint.PrettyPrinter(indent=2)

print("Available Record Sets in metadata.recordSet:")
if hasattr(metadata, 'recordSet'):
    for rs in metadata.recordSet:
        print(f"- RecordSet name: {getattr(rs, 'name', None)}   @id: {getattr(rs, '@id', None)}")
        if hasattr(rs, 'field'):
            print("  Fields:")
            for field in rs.field:
                print(f"    - {getattr(field, 'name', None)}   @id: {getattr(field, '@id', None)}   dataType: {getattr(field, 'dataType', None)}")
        # List columns in each field
        if hasattr(rs, 'columns'):
            print("  Columns:")
            for col in rs.columns:
                print(f"    - {getattr(col, 'name', None)}   @id: {getattr(col, '@id', None)}")
else:
    print("No recordSet found in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis using the record set and field `@id`s discovered above.

In [ ]:
# For this dataset, let's list all recordSet @ids discovered above.

# Example for how to use the first RecordSet (or substitute your target @id here):
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        if hasattr(rs, '@id'):
            record_sets.append(getattr(rs, '@id'))
else:
    print("No record sets found.")

dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"Loaded DataFrame for record set {record_set}, shape: {df.shape}")

# For further exploration, select the first record set
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"\nColumns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering, normalizing numeric fields, or grouping.

In [ ]:
# Let's attempt numeric analysis on a likely numeric field.
# Manually examine the columns; if none, modify as appropriate for your actual dataset.
import numpy as np

numeric_field = None
group_field = None

df = dataframes.get(main_record_set_id, pd.DataFrame())
if not df.empty:
    print("\nAttempting to select a numeric field (e.g., 'MW', 'Coverage %', etc.)...")
    # Try to guess a numeric field from columns
    for col in df.columns:
        if df[col].dtype == float or df[col].dtype == int:
            numeric_field = col
            break
    if not numeric_field:
        potential_numeric = [col for col in df.columns if any(x in col.lower() for x in ['mw', 'coverage', 'count', 'abundance'])]
        if potential_numeric:
            numeric_field = potential_numeric[0]
    if numeric_field:
        # Make sure column is numeric type
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

        threshold = df[numeric_field].quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}: (showing top 5)")
        display(filtered_df.head())

        # Normalize the field
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records (showing top 5):")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Find a grouping field
        # Prefer a non-numeric field that's not a system field
        for col in df.columns:
            if df[col].dtype == object and col != numeric_field and not col.startswith('_') and col != norm_col:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).to_frame()
            print(f"\nGrouped mean {numeric_field} by '{group_field}' (showing top 5):")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric group field found.")
    else:
        print("No numeric fields found to analyze.")
else:
    print("No DataFrame to analyze.")

## 5. Visualization
Visualize the distribution of a numeric field or relationships if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution if possible
if not df.empty and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df[[group_field, numeric_field]].dropna(), x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we've loaded and explored the extracellular vesicle protein mass spectrometry dataset using the Croissant schema via `mlcroissant`. We reviewed record sets and fields by their `@id`s, extracted data into DataFrames, performed simple EDA including filtering and normalization on numeric fields, grouped by categorical attributes, and visualized data distributions.

Further in-depth analyses—including advanced statistical modeling, biological interpretation, or integration with other omics datasets—are possible using this workflow and the transparent record and field referencing provided by Croissant.

**Remember:** All dataset entities are referenced and traced via their Croissant `@id` for reproducibility and FAIR data access.